# 11 — โครงงานย่อย: วิเคราะห์ฝนรายวันจังหวัดชลบุรี ปี 2025

**คอร์ส Python for Geography** · ภาควิชาภูมิศาสตร์ ชั้นปีที่ 2
**วันที่ 6 · ช่วงที่ 2 · เวลาโดยประมาณ 60 นาที + งานส่ง**

---

### โจทย์

คุณได้รับข้อมูลฝนรายวันของ 4 สถานีในจังหวัดชลบุรี ตลอดปี 2025
ให้เขียนโปรแกรมตอบคำถามต่อไปนี้ แล้วสรุปเป็นรายงานสั้น ๆ พร้อมกราฟ

1. สถานีใดมีฝนรวมทั้งปีมากที่สุด/น้อยที่สุด ต่างกันเท่าไร
2. ฤดูฝนของพื้นที่นี้เริ่มและสิ้นสุดเดือนใด (ดูจากฝนรายเดือน)
3. มีวันที่ "ฝนหนักมาก" (> 90 mm) กี่วัน เกิดที่ไหน เมื่อไร
4. ฝนสะสม 3 วันสูงสุดของแต่ละสถานีเป็นเท่าไร เกิดช่วงใด
5. สถานีที่อยู่ห่างจากชายฝั่งมากกว่า มีฝนมากกว่าหรือน้อยกว่า — ให้ใช้ข้อมูลสนับสนุนคำตอบ

### สิ่งที่ต้องส่ง
- notebook ที่รันได้ตั้งแต่ต้นจนจบ (Runtime → Restart and run all ต้องผ่าน)
- กราฟอย่างน้อย 3 รูป มีชื่อกราฟ ชื่อแกน และหน่วยครบ
- ไฟล์ `summary_<รหัสนักศึกษา>.csv` ที่สรุปผลรายสถานี
- ข้อสรุปเป็นข้อความ 5–10 บรรทัด ใน text cell ท้าย notebook

### เกณฑ์ให้คะแนน (100)
| หัวข้อ | คะแนน |
|---|---|
| โค้ดรันได้ครบไม่มี error | 20 |
| ตอบคำถามครบทั้ง 5 ข้อ ถูกต้อง | 30 |
| คุณภาพกราฟ (ครบองค์ประกอบ อ่านง่าย) | 20 |
| การใช้ฟังก์ชัน/โครงสร้างโค้ดที่สะอาด มี comment | 15 |
| การตีความผลเชิงภูมิศาสตร์ | 15 |

## ขั้นที่ 0 — เตรียมข้อมูลและ import

In [ ]:
# ===== เตรียมไฟล์ข้อมูลสำหรับบทนี้ (รันครั้งเดียวต่อ session) =====
# ถ้าอาจารย์เปลี่ยนชื่อ repo ให้แก้ URL บรรทัดล่างนี้บรรทัดเดียว
BASE = "https://raw.githubusercontent.com/jamorn12/Teaching_Python_for_Geography/main/"

import os, subprocess, sys, urllib.request

FILES = ["stations.csv", "rainfall_daily_2025.csv", "messy_rainfall.csv"]
os.makedirs("data", exist_ok=True)


def have_all():
    return all(os.path.exists(os.path.join("data", n)) for n in FILES)


# 1) ถ้ามีไฟล์อยู่แล้ว (เช่น clone repo มา หรืออัปโหลดเอง) ก็ใช้ได้เลย
if not have_all():
    # 2) ลองดาวน์โหลดจาก GitHub
    for name in FILES:
        target = os.path.join("data", name)
        if not os.path.exists(target):
            try:
                urllib.request.urlretrieve(BASE + "data/" + name, target)
            except Exception:
                pass

if not have_all():
    # 3) สำรองสุดท้าย: สร้างข้อมูลชุดเดียวกันขึ้นมาใหม่จาก make_datasets.py
    script = None
    for cand in ["make_datasets.py", "utils/make_datasets.py"]:
        if os.path.exists(cand):
            script = cand
            break
    if script is None:
        try:
            urllib.request.urlretrieve(BASE + "utils/make_datasets.py", "make_datasets.py")
            script = "make_datasets.py"
        except Exception:
            script = None
    if script:
        subprocess.run([sys.executable, script, "--outdir", "data"], check=True)

if have_all():
    print("ข้อมูลพร้อมใช้งาน:", sorted(os.listdir("data")))
else:
    print("ยังไม่พบข้อมูล -> ให้ upload โฟลเดอร์ data/ เข้ามา "
          "หรือรัน utils/make_datasets.py เพื่อสร้างข้อมูลใหม่")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os, urllib.request

try:
    p = "Sarabun-Regular.ttf"
    if not os.path.exists(p):
        urllib.request.urlretrieve(
            "https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf", p)
    fm.fontManager.addfont(p)
    matplotlib.rc("font", family="Sarabun")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as exc:
    print("ฟอนต์ไทยไม่พร้อม:", exc)

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110
print("พร้อมทำงาน")

## ขั้นที่ 1 — โหลดและตรวจสอบข้อมูล

**สิ่งที่ต้องทำ:** อ่าน `data/rainfall_daily_2025.csv` และ `data/stations.csv`
แปลงคอลัมน์ `date` เป็น datetime เพิ่มคอลัมน์ `month`
แล้วตรวจว่าข้อมูลครบทุกสถานี ทุกวันหรือไม่

In [ ]:
# ✍️ เขียนโค้ดขั้นที่ 1 ที่นี่

## ขั้นที่ 2 — สรุปรายสถานี (ตอบคำถามข้อ 1)

**สิ่งที่ต้องทำ:** ใช้ `groupby().agg()` สร้างตารางสรุปที่มี
`total_mm`, `max_daily_mm`, `mean_daily_mm`, `rain_days`
แล้วหาสถานีที่ฝนมากที่สุดและน้อยที่สุด

In [ ]:
# ✍️ เขียนโค้ดขั้นที่ 2 ที่นี่

## ขั้นที่ 3 — ฝนรายเดือนและฤดูฝน (ตอบคำถามข้อ 2)

**สิ่งที่ต้องทำ:** สร้างตารางฝนรายเดือน × สถานี (ใบ้: `pivot_table`)
แล้วพล็อตกราฟเส้นหรือแท่งเปรียบเทียบ
นิยามอย่างง่ายของ "เดือนในฤดูฝน" ให้ใช้เกณฑ์ฝนรวมเดือนนั้น ≥ 100 mm

In [ ]:
# ✍️ เขียนโค้ดขั้นที่ 3 ที่นี่

## ขั้นที่ 4 — เหตุการณ์ฝนหนักมาก (ตอบคำถามข้อ 3)

**สิ่งที่ต้องทำ:** กรองวันที่ฝน > 90 mm แสดงวันที่ สถานี และปริมาณ เรียงจากมากไปน้อย

In [ ]:
# ✍️ เขียนโค้ดขั้นที่ 4 ที่นี่

## ขั้นที่ 5 — ฝนสะสม 3 วัน (ตอบคำถามข้อ 4)

**สิ่งที่ต้องทำ:** เขียน **ฟังก์ชัน** `max_3day(df, station_id)`
ที่คืน `(วันที่สิ้นสุด, ค่าสะสม)` แล้วเรียกใช้กับทุกสถานี
(ใบ้: เรียงตามวันที่ก่อน แล้วใช้ `.rolling(3).sum()`)

In [ ]:
# ✍️ เขียนโค้ดขั้นที่ 5 ที่นี่

## ขั้นที่ 6 — ระยะห่างจากชายฝั่งกับปริมาณฝน (ตอบคำถามข้อ 5)

**สิ่งที่ต้องทำ:** ใช้ฟังก์ชัน `haversine` จากบทที่ 07 คำนวณระยะจากสถานีเกาะสีชัง
(`SRC01` ซึ่งเป็นจุดอ้างอิงฝั่งทะเล) ไปยังแต่ละสถานี แล้วพล็อต scatter
ระหว่างระยะทางกับฝนรวมทั้งปี พร้อมคำนวณค่าสหสัมพันธ์

In [ ]:
# ✍️ เขียนโค้ดขั้นที่ 6 ที่นี่

## ขั้นที่ 7 — บันทึกผลและเขียนข้อสรุป

**สิ่งที่ต้องทำ:** บันทึกตารางสรุปเป็น `summary_<รหัสนักศึกษา>.csv`
แล้วเขียนข้อสรุป 5–10 บรรทัดใน text cell ด้านล่าง

In [ ]:
# ✍️ เขียนโค้ดขั้นที่ 7 ที่นี่

### ✍️ ข้อสรุปของฉัน

(เขียนที่นี่ — อธิบายว่าพบอะไร และอธิบายด้วยเหตุผลเชิงภูมิศาสตร์ว่าทำไมถึงเป็นเช่นนั้น)

---

---
# 🔑 เฉลยอ้างอิง (สำหรับผู้สอน)

> **ผู้สอน:** ลบเซลล์ตั้งแต่บรรทัดนี้ลงไปก่อนแจกให้นักศึกษา
> หรือใช้เป็นเฉลยเฉลยหลังหมดเวลาทำ

In [ ]:
# ---------- ขั้นที่ 1 ----------
df = pd.read_csv("data/rainfall_daily_2025.csv")
stations = pd.read_csv("data/stations.csv")
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month
df["is_rain"] = df["rain_mm"] >= 0.1

print(df.shape)
print(df.groupby("station_id").size())
print("ค่าที่หายไป:", df.isna().sum().sum())
df.head()

In [ ]:
# ---------- ขั้นที่ 2 ----------
summary = df.groupby("station_id").agg(
    total_mm=("rain_mm", "sum"),
    max_daily_mm=("rain_mm", "max"),
    mean_daily_mm=("rain_mm", "mean"),
    rain_days=("is_rain", "sum"),
).round(2).reset_index()

summary = summary.merge(stations, on="station_id")
wet = summary.loc[summary["total_mm"].idxmax()]
dry = summary.loc[summary["total_mm"].idxmin()]
print(f"ฝนมากสุด : {wet['station_id']} ({wet['station_name']}) {wet['total_mm']:.1f} mm")
print(f"ฝนน้อยสุด: {dry['station_id']} ({dry['station_name']}) {dry['total_mm']:.1f} mm")
print(f"ต่างกัน   : {wet['total_mm'] - dry['total_mm']:.1f} mm "
      f"({(wet['total_mm']/dry['total_mm'] - 1) * 100:.0f}%)")
summary[["station_id", "station_name", "total_mm", "max_daily_mm", "rain_days"]]

In [ ]:
# ---------- ขั้นที่ 3 ----------
monthly = df.pivot_table(index="month", columns="station_id",
                         values="rain_mm", aggfunc="sum").round(1)
display(monthly)

monthly.plot(marker="o")
plt.title("ฝนรวมรายเดือน 4 สถานี จังหวัดชลบุรี ปี 2025")
plt.xlabel("เดือน"); plt.ylabel("ปริมาณฝนรวม (mm)")
plt.xticks(range(1, 13)); plt.grid(alpha=0.3)
plt.legend(title="สถานี"); plt.show()

wet_months = monthly.mean(axis=1)[monthly.mean(axis=1) >= 100].index.tolist()
print("เดือนในฤดูฝน (เฉลี่ยทุกสถานี >= 100 mm):", wet_months)

In [ ]:
# ---------- ขั้นที่ 4 ----------
extreme = df[df["rain_mm"] > 90].sort_values("rain_mm", ascending=False)
print(f"พบวันฝนหนักมาก {len(extreme)} วัน")
extreme[["date", "station_id", "station_name", "rain_mm"]]

In [ ]:
# ---------- ขั้นที่ 5 ----------
def max_3day(data, station_id):
    """คืน (วันที่สิ้นสุด, ฝนสะสม 3 วันสูงสุด) ของสถานีที่ระบุ"""
    sub = data[data["station_id"] == station_id].sort_values("date").copy()
    sub["acc3"] = sub["rain_mm"].rolling(3).sum()
    row = sub.loc[sub["acc3"].idxmax()]
    return row["date"].date(), round(row["acc3"], 1)

for sid in summary["station_id"]:
    end_date, value = max_3day(df, sid)
    print(f"{sid}: {value:6.1f} mm สิ้นสุดวันที่ {end_date}")

In [ ]:
# ---------- ขั้นที่ 6 ----------
import math

def haversine(lat1, lon1, lat2, lon2):
    """ระยะทางวงกลมใหญ่ระหว่างสองพิกัด (km)"""
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlam/2)**2
    return 2 * R * math.asin(math.sqrt(a))

ref = stations[stations["station_id"] == "SRC01"].iloc[0]
summary["dist_from_sea_km"] = summary.apply(
    lambda r: round(haversine(ref["lat"], ref["lon"], r["lat"], r["lon"]), 2), axis=1)

plt.figure(figsize=(6.5, 5))
plt.scatter(summary["dist_from_sea_km"], summary["total_mm"], s=90, color="#3b7dd8", edgecolor="black")
for _, r in summary.iterrows():
    plt.text(r["dist_from_sea_km"] + 1, r["total_mm"], r["station_id"], fontsize=9)
plt.title("ระยะห่างจากทะเล กับ ปริมาณฝนรวมทั้งปี")
plt.xlabel("ระยะจากสถานีเกาะสีชัง (km)"); plt.ylabel("ฝนรวมทั้งปี (mm)")
plt.grid(alpha=0.3); plt.show()

r = summary["dist_from_sea_km"].corr(summary["total_mm"])
print(f"สหสัมพันธ์ r = {r:.3f}")
summary[["station_id", "station_name", "elevation_m", "dist_from_sea_km", "total_mm"]]

In [ ]:
# ---------- ขั้นที่ 7 ----------
summary.to_csv("summary_reference.csv", index=False, encoding="utf-8-sig")
print("บันทึกไฟล์แล้ว")
!head -3 summary_reference.csv

### ข้อสรุปอ้างอิง

- สถานีหนองใหญ่ (NYI01) ซึ่งอยู่ลึกเข้าไปในแผ่นดินและสูงกว่าสถานีอื่น มีฝนรวมทั้งปีสูงสุด
  ส่วนเกาะสีชัง (SRC01) ซึ่งเป็นเกาะกลางทะเล มีฝนรวมต่ำสุด
- ฤดูฝนของพื้นที่นี้ชัดเจนในช่วงเดือนพฤษภาคมถึงตุลาคม สอดคล้องกับอิทธิพลของ
  southwest monsoon และมีจุดสูงสุดในเดือนกันยายน
- วันฝนหนักมาก (> 90 mm) กระจุกตัวอยู่ในเดือนสิงหาคม–ตุลาคม ซึ่งเป็นช่วงที่ทั้งมรสุม
  และหย่อมความกดอากาศต่ำมีโอกาสเกิดพร้อมกัน
- ค่าสหสัมพันธ์ระหว่างระยะห่างจากทะเลกับฝนรวมเป็นบวก แต่ด้วยจำนวนสถานีเพียง 4 แห่ง
  **ยังสรุปเชิงสถิติไม่ได้** ควรระวังการตีความจากกลุ่มตัวอย่างขนาดเล็ก
- ข้อควรระวัง: ชุดข้อมูลนี้เป็นข้อมูลสังเคราะห์เพื่อการสอน ไม่ใช่ข้อมูลตรวจวัดจริง
  ข้อสรุปทั้งหมดจึงใช้ฝึกกระบวนการวิเคราะห์เท่านั้น ไม่ควรนำไปอ้างอิงทางวิชาการ